In [1]:
!pip install -q transformers datasets accelerate scikit-learn pandas

In [1]:
import os
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

# =========================
# 1. 설정
# =========================

DATA_PATH = r"C:\Users\wanje\Desktop\deeplearning\Raw\mosi_text_metadata.csv"   # 본인 csv 경로로 수정
MODEL_NAME = "roberta-base"
MAX_LEN = 70
OUTPUT_DIR = r"C:\Users\wanje\Desktop\deeplearning\roberta_mosi_text"  # 모델 저장 경로로 수정

# =========================
# 2. 데이터 불러오기
# =========================

df = pd.read_csv(DATA_PATH)

print(df.head())
print(df.columns)

# text 컬럼 이름 자동 탐색
possible_text_cols = ["text", "Text", "sentence", "utterance", "transcript"]
text_col = None

for col in possible_text_cols:
    if col in df.columns:
        text_col = col
        break

if text_col is None:
    raise ValueError("text 컬럼을 찾지 못했습니다. CSV의 텍스트 컬럼명을 확인하세요.")

df = df.dropna(subset=[text_col]).reset_index(drop=True)
df[text_col] = df[text_col].astype(str)

# =========================
# 3. 라벨 만들기
# =========================
# CMU-MOSI는 보통 sentiment 값이 -3 ~ +3 범위임
# 여기서는 3분류로 변환:
# negative = 0, neutral = 1, positive = 2

def make_label(row):
    # 이미 label 컬럼이 있는 경우
    if "label" in row.index:
        value = row["label"]

        # 문자열 라벨 처리
        if isinstance(value, str):
            value = value.lower().strip()
            if value in ["n", "neg", "negative"]:
                return 0
            elif value in ["neutral", "neu"]:
                return 1
            elif value in ["p", "pos", "positive"]:
                return 2

        # 숫자 라벨 처리: -1, 0, 1 형태
        try:
            value = int(value)
            if value == -1:
                return 0
            elif value == 0:
                return 1
            elif value == 1:
                return 2
            elif value == 2:
                return 2
        except:
            pass

    # sentiment 컬럼이 있는 경우
    if "sentiment" in row.index:
        score = float(row["sentiment"])

        if score < 0:
            return 0      # negative
        elif score == 0:
            return 1      # neutral
        else:
            return 2      # positive

    raise ValueError("label 또는 sentiment 컬럼이 필요합니다.")


df["label_id"] = df.apply(make_label, axis=1)

print(df[[text_col, "label_id"]].head())
print(df["label_id"].value_counts())

# 혹시 특정 라벨이 없는 경우에도 처리 가능하게 라벨을 다시 0부터 정리
unique_labels = sorted(df["label_id"].unique())
label_remap = {old: new for new, old in enumerate(unique_labels)}
df["label_id"] = df["label_id"].map(label_remap)

num_labels = len(unique_labels)

print("라벨 개수:", num_labels)
print("라벨 매핑:", label_remap)

id2label = {}
for old, new in label_remap.items():
    if old == 0:
        id2label[new] = "negative"
    elif old == 1:
        id2label[new] = "neutral"
    elif old == 2:
        id2label[new] = "positive"
    else:
        id2label[new] = str(old)

label2id = {v: k for k, v in id2label.items()}

# =========================
# 4. train / valid / test split
# =========================
# video_id가 있으면 같은 영상의 segment가 train/test에 섞이지 않게 group split 사용

if "video_id" in df.columns:
    groups = df["video_id"]

    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    train_idx, temp_idx = next(gss.split(df, df["label_id"], groups))

    train_df = df.iloc[train_idx].reset_index(drop=True)
    temp_df = df.iloc[temp_idx].reset_index(drop=True)

    temp_groups = temp_df["video_id"]
    gss2 = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=42)
    valid_idx, test_idx = next(gss2.split(temp_df, temp_df["label_id"], temp_groups))

    valid_df = temp_df.iloc[valid_idx].reset_index(drop=True)
    test_df = temp_df.iloc[test_idx].reset_index(drop=True)

else:
    train_df, temp_df = train_test_split(
        df,
        test_size=0.2,
        random_state=42,
        stratify=df["label_id"]
    )

    valid_df, test_df = train_test_split(
        temp_df,
        test_size=0.5,
        random_state=42,
        stratify=temp_df["label_id"]
    )

print("train:", len(train_df))
print("valid:", len(valid_df))
print("test:", len(test_df))

# =========================
# 5. Hugging Face Dataset 변환
# =========================

train_dataset = Dataset.from_pandas(train_df[[text_col, "label_id"]])
valid_dataset = Dataset.from_pandas(valid_df[[text_col, "label_id"]])
test_dataset = Dataset.from_pandas(test_df[[text_col, "label_id"]])

# =========================
# 6. Tokenizer
# =========================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(batch):
    return tokenizer(
        batch[text_col],
        padding="max_length",
        truncation=True,
        max_length=MAX_LEN
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
valid_dataset = valid_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

train_dataset = train_dataset.rename_column("label_id", "labels")
valid_dataset = valid_dataset.rename_column("label_id", "labels")
test_dataset = test_dataset.rename_column("label_id", "labels")

train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
valid_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

# =========================
# 7. RoBERTa 분류 모델
# =========================

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

# =========================
# 8. 평가 지표
# =========================

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        average="weighted",
        zero_division=0
    )

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

# =========================
# 9. 학습 설정
# =========================

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none"
)

# 만약 eval_strategy에서 에러가 나면,
# eval_strategy="epoch"를 evaluation_strategy="epoch"로 바꾸면 됨.

# =========================
# 10. Trainer
# =========================

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

# =========================
# 11. 학습
# =========================

trainer.train()

# =========================
# 12. 테스트 평가
# =========================

test_result = trainer.evaluate(test_dataset)
print(test_result)

pred_output = trainer.predict(test_dataset)
preds = np.argmax(pred_output.predictions, axis=-1)
true_labels = pred_output.label_ids

print(classification_report(
    true_labels,
    preds,
    target_names=[id2label[i] for i in range(num_labels)],
    zero_division=0
))

# =========================
# 13. 모델 저장
# =========================

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("모델 저장 완료:", OUTPUT_DIR)

c:\Users\wanje\anaconda3\envs\main\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ValueError: Your currently installed version of Keras is Keras 3, but this is not yet supported in Transformers. Please install the backwards-compatible tf-keras package with `pip install tf-keras`.

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModel, AutoTokenizer

class RobertaTextEncoder256(nn.Module):
    def __init__(self, model_name="roberta-base", output_dim=256, dropout=0.3):
        super().__init__()

        self.roberta = AutoModel.from_pretrained(model_name)

        hidden_size = self.roberta.config.hidden_size  # roberta-base는 보통 768

        self.projection = nn.Sequential(
            nn.Linear(hidden_size, output_dim),
            nn.LayerNorm(output_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.roberta(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # RoBERTa의 첫 번째 토큰 feature 사용
        cls_feature = outputs.last_hidden_state[:, 0, :]

        # 768차원 → 256차원
        text_feature = self.projection(cls_feature)

        return text_feature
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained("roberta-base")
text_encoder = RobertaTextEncoder256(
    model_name="roberta-base",
    output_dim=256
).to(device)

sample_texts = [
    "I really liked this movie.",
    "It was disappointing and boring."
]

encoded = tokenizer(
    sample_texts,
    padding=True,
    truncation=True,
    max_length=70,
    return_tensors="pt"
)

input_ids = encoded["input_ids"].to(device)
attention_mask = encoded["attention_mask"].to(device)

text_feature = text_encoder(input_ids, attention_mask)

print(text_feature.shape)